# EDA — Retail Rocket events

Notebook khám phá `events_sorted.csv`: cấu trúc, chất lượng dữ liệu, hành vi visitor/item và xu hướng thời gian. Dataset gốc chỉ được đọc, không chỉnh sửa.

> `visitorid` là visitor ID trong dataset, không phải session ID hay danh tính khách hàng đã xác minh. Các thống kê funnel bên dưới tính theo visitor, không phải session. Timestamp được chuyển sang UTC.

## 1. Ý nghĩa các trường dữ liệu

Mỗi dòng là một event của visitor trên website. Dataset có năm trường gốc:

| Trường | Ý nghĩa | Lưu ý khi phân tích |
|---|---|---|
| `timestamp` | Thời điểm event xảy ra, Unix epoch tính bằng milliseconds. | Đây là Event Time; notebook đổi sang UTC để vẽ và tổng hợp theo ngày/giờ. |
| `visitorid` | Mã visitor do Retail Rocket gán trong dữ liệu. | Là khóa để nối các event cùng visitor trong dataset; không phải thông tin định danh cá nhân đã xác minh và cũng không phải session ID. |
| `event` | Loại hành động: `view`, `addtocart` hoặc `transaction`. | Một visitor có thể có nhiều dòng và nhiều loại event. |
| `itemid` | Mã sản phẩm gắn với event. | Dùng để phân tích lượt xem, thêm giỏ và giao dịch theo sản phẩm. |
| `transactionid` | Mã giao dịch; thường chỉ có giá trị với event `transaction`. | Giá trị trống ở các event khác là bình thường, không nhất thiết là dữ liệu thiếu lỗi. |

Retail Rocket không cung cấp session ID trong schema này. Vì vậy, các phân tích theo `visitorid` trong notebook là theo visitor ID của dataset, không thể kết luận chúng tương ứng chính xác với từng phiên truy cập hay một khách hàng thật.

## 2. Cài thư viện và đọc dữ liệu
Nếu môi trường chưa có thư viện: `python -m pip install pandas matplotlib jupyter`. Dùng dtype nhỏ để giảm bộ nhớ khi đọc khoảng 2.75 triệu dòng.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = Path('../data/events_sorted.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/events_sorted.csv')  # fallback nếu chạy notebook từ thư mục repository

DTYPES = {
    'visitorid': 'int32',
    'event': 'category',
    'itemid': 'int32',
    'transactionid': 'Int32',
}
df = pd.read_csv(DATA_PATH, dtype=DTYPES)
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
df['event_time'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True, errors='coerce')
print(f'File: {DATA_PATH.resolve()}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MiB')
display(df.head())

## 3. Schema và chất lượng dữ liệu

In [ ]:
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': (df.isna().mean() * 100).round(4)}))
print('Event labels:', df['event'].value_counts(dropna=False).to_dict())
print('Timestamp range (UTC):', df['event_time'].min(), '→', df['event_time'].max())
print('Rows with invalid timestamps:', int(df['event_time'].isna().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Negative IDs:', {c: int((df[c].dropna() < 0).sum()) for c in ['visitorid', 'itemid', 'transactionid']})
print('Transaction ID on non-transaction rows:', int(df.loc[df.event != 'transaction', 'transactionid'].notna().sum()))
print('Transaction rows without transaction ID:', int(df.loc[df.event == 'transaction', 'transactionid'].isna().sum()))

## 4. Phân bố loại sự kiện
Tỷ lệ được tính trên số event rows, không phải số visitor hoặc session.

In [ ]:
event_counts = df['event'].value_counts().rename_axis('event').to_frame('events')
event_counts['share_pct'] = (event_counts['events'] / len(df) * 100).round(2)
display(event_counts)
ax = event_counts['events'].sort_values(ascending=False).plot.bar(figsize=(8, 4), color='#3478a8')
ax.set(title='Retail Rocket events by type', xlabel='Event', ylabel='Rows')
ax.ticklabel_format(axis='y', style='plain')
plt.tight_layout()
plt.show()

## 5. Visitor và item
Các phân vị cho thấy mức độ lệch giữa visitor hoạt động ít và visitor hoạt động nhiều. `visitorid` không xác định session.

In [ ]:
visitor_events = df.groupby('visitorid', observed=True).size().rename('events')
item_events = df.groupby('itemid', observed=True).size().rename('events')
visitor_items = df.groupby('visitorid', observed=True)['itemid'].nunique().rename('unique_items')
print(f'Unique visitors: {df.visitorid.nunique():,}')
print(f'Unique items: {df.itemid.nunique():,}')
print('Events per visitor quantiles:')
display(visitor_events.quantile([0, .25, .5, .75, .9, .95, .99, 1]).to_frame())
print('Unique items per visitor quantiles:')
display(visitor_items.quantile([0, .25, .5, .75, .9, .95, .99, 1]).to_frame())
print('Top visitors by event count:')
display(visitor_events.nlargest(10).to_frame())
print('Top items by event count:')
display(item_events.nlargest(10).to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
visitor_events.clip(upper=visitor_events.quantile(.99)).hist(bins=50, ax=axes[0], color="#52a675")
axes[0].set(title="Events per visitor (clipped at p99)", xlabel="Events", ylabel="Visitors")
item_events.clip(upper=item_events.quantile(.99)).hist(bins=50, ax=axes[1], color="#d28b44")
axes[1].set(title="Events per item (clipped at p99)", xlabel="Events", ylabel="Items")
plt.tight_layout()
plt.show()

## 6. Phân tích phiên truy cập suy ra từ visitor events

Dataset không có session ID. Ở đây ta **ước lượng** session theo quy tắc: sắp xếp event theo thời gian trong từng `visitorid`; nếu khoảng cách từ event trước đó lớn hơn 30 phút thì bắt đầu session mới. Ngưỡng 30 phút là giả định phân tích phổ biến, không phải nhãn session có sẵn trong Retail Rocket. Có thể đổi `SESSION_GAP_MINUTES` để xem kết quả thay đổi.

Đây là cách hữu ích để hiểu sessionization trong streaming. Nó khác sliding window cố định 10 phút của pipeline: session window được neo theo hoạt động của visitor và kết thúc sau khoảng im lặng; pipeline hiện tại **chưa** dùng session window. Event đến trễ có thể làm thay đổi cách chia phiên, nên Spark streaming cần watermark/state để biết khi nào có thể chốt và dọn state.

Khoảng cách giữa hai event trong cùng phiên chỉ là inter-event gap, không khẳng định visitor thực sự nhìn màn hình trong suốt khoảng đó. Phiên ở đầu/cuối dataset cũng có thể bị cắt cụt.

In [ ]:
SESSION_GAP_MINUTES = 30
SESSION_GAP = pd.Timedelta(minutes=SESSION_GAP_MINUTES)

# Sessionize independently per visitor using event time, not file processing time.
session_events = (df.dropna(subset=["event_time"])
                  .sort_values(["visitorid", "event_time", "timestamp"], kind="mergesort")
                  .copy())
session_events["previous_event_time"] = session_events.groupby("visitorid", sort=False)["event_time"].shift()
session_events["gap_from_previous_seconds"] = (session_events["event_time"] - session_events["previous_event_time"]).dt.total_seconds()
session_events["new_session"] = (session_events["previous_event_time"].isna() |
                                 (session_events["event_time"] - session_events["previous_event_time"] > SESSION_GAP))
session_events["session_number"] = session_events.groupby("visitorid", sort=False)["new_session"].cumsum().astype("int32")
session_events["gap_inside_session_seconds"] = session_events["gap_from_previous_seconds"].where(~session_events["new_session"])
for event_name, column in [("view", "views"), ("addtocart", "addtocarts"), ("transaction", "transactions")]:
    session_events[column] = (session_events["event"] == event_name).astype("uint8")

session_keys = ["visitorid", "session_number"]
# Built-in reductions avoid running Python callbacks once per session.
session_summary = (session_events.groupby(session_keys, sort=False, observed=True)
    .agg(start_time=("event_time", "min"), end_time=("event_time", "max"),
         event_count=("event", "size"), unique_items=("itemid", "nunique"),
         views=("views", "sum"), addtocarts=("addtocarts", "sum"),
         transactions=("transactions", "sum"))
    .reset_index())
session_summary["duration_seconds"] = (session_summary["end_time"] - session_summary["start_time"]).dt.total_seconds()
session_summary["duration_minutes"] = session_summary["duration_seconds"] / 60
session_summary["contains_cart"] = session_summary["addtocarts"] > 0
session_summary["contains_transaction"] = session_summary["transactions"] > 0

print(f"Session gap threshold: {SESSION_GAP_MINUTES} minutes")
print(f"Derived sessions: {len(session_summary):,}")
print(f"Visitors with events: {session_summary.visitorid.nunique():,}")
display(session_summary[["duration_seconds", "duration_minutes", "event_count", "unique_items", "views", "addtocarts", "transactions"]].describe(percentiles=[.5,.75,.9,.95,.99]).T)
print("Sessions with one event (duration is zero):", int((session_summary.event_count == 1).sum()))
print("Sessions containing cart:", int(session_summary.contains_cart.sum()))
print("Sessions containing transaction:", int(session_summary.contains_transaction.sum()))

### Phiên dài nhất, ngắn nhất và event timeline
Phiên một event có duration bằng 0 vì không có hai mốc thời gian để tính độ dài. Vì vậy, phần dưới hiển thị cả phiên ngắn nhất theo định nghĩa (0 phút nếu single-event session) và phiên ngắn nhất có duration dương.

In [ ]:
longest = session_summary.loc[session_summary.duration_seconds.idxmax()]
shortest = session_summary.loc[session_summary.duration_seconds.idxmin()]
positive_sessions = session_summary[session_summary.duration_seconds > 0]
shortest_positive = (positive_sessions.loc[positive_sessions.duration_seconds.idxmin()]
                    if not positive_sessions.empty else None)

def print_session(label, row):
    if row is None:
        print(label + ": không có session có duration dương")
        return
    print(f"{label}: visitor={int(row.visitorid)}, session={int(row.session_number)}, "
          f"duration={row.duration_seconds:.3f}s ({row.duration_minutes:.3f}m), "
          f"events={int(row.event_count)}, items={int(row.unique_items)}, "
          f"views={int(row.views)}, addtocarts={int(row.addtocarts)}, transactions={int(row.transactions)}")
    print(f"  start={row.start_time}; end={row.end_time}")

print_session("Longest session", longest)
print_session("Shortest session", shortest)
print_session("Shortest non-zero session", shortest_positive)

def show_session_timeline(row, label):
    if row is None: return
    mask = ((session_events.visitorid == row.visitorid) &
            (session_events.session_number == row.session_number))
    timeline = session_events.loc[mask, ["event_time", "event", "itemid", "transactionid",
                                          "gap_from_previous_seconds", "gap_inside_session_seconds"]].copy()
    timeline.insert(0, "event_order", range(1, len(timeline) + 1))
    print(label, "— every event and its timestamp:")
    display(timeline.reset_index(drop=True))

show_session_timeline(longest, "Longest session")
show_session_timeline(shortest, "Shortest session")
if shortest_positive is not None and shortest_positive.session_number != shortest.session_number:
    show_session_timeline(shortest_positive, "Shortest positive-duration session")

### Phân bố session, event mix và thời điểm bắt đầu

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
session_summary.duration_minutes.clip(upper=session_summary.duration_minutes.quantile(.99)).hist(bins=50, ax=axes[0], color="#7057a8")
axes[0].set(title="Session duration (clipped at p99)", xlabel="Minutes", ylabel="Sessions")
session_summary.event_count.clip(upper=session_summary.event_count.quantile(.99)).hist(bins=50, ax=axes[1], color="#3a9b91")
axes[1].set(title="Events per session (clipped at p99)", xlabel="Events", ylabel="Sessions")
plt.tight_layout(); plt.show()

# Event mix per derived session: bit 1=view, bit 2=cart, bit 4=transaction.
mix_code = (session_summary.views.gt(0).astype("int8") +
            2 * session_summary.addtocarts.gt(0).astype("int8") +
            4 * session_summary.transactions.gt(0).astype("int8"))
mix_labels = {1:"view only", 2:"cart only", 3:"view + cart", 4:"transaction only",
              5:"view + transaction", 6:"cart + transaction", 7:"view + cart + transaction"}
event_mix = mix_code.map(mix_labels).value_counts().rename_axis("event_mix").to_frame("sessions")
display(event_mix)

sessions_per_visitor = session_summary.groupby("visitorid", observed=True).size().rename("sessions")
print("Sessions per visitor quantiles:")
display(sessions_per_visitor.quantile([0,.25,.5,.75,.9,.95,.99,1]).to_frame())

session_start_hour = session_summary.start_time.dt.hour.value_counts().sort_index().reindex(range(24), fill_value=0)
session_start_hour.plot.bar(figsize=(10,3), color="#3478a8")
plt.title("Derived session starts by hour (UTC)"); plt.xlabel("Hour (UTC)"); plt.ylabel("Sessions")
plt.tight_layout(); plt.show()

### Sessionization trong Spark Streaming
Trong Structured Streaming, sessionization thường dùng Spark `session_window` theo visitor và gap duration, cùng watermark để giới hạn state. Khi có event đến muộn, hai vùng hoạt động có thể nối thành một session; vì vậy ranh giới session có thể thay đổi trước khi watermark cho phép chốt state. Ngược lại, pipeline hiện tại group theo fixed sliding window 10 phút / slide 5 phút — nó không suy ra phiên người dùng. EDA này là phân tích batch có thể sắp xếp toàn bộ lịch sử; streaming chỉ nhìn thấy event đã đến đến thời điểm hiện tại.

## 7. Xu hướng theo thời gian
Biểu đồ dùng UTC. Gaps toàn cục giữa hai dòng liên tiếp không phải session duration vì file chứa nhiều visitor xen kẽ.

In [ ]:
valid_time = df.dropna(subset=['event_time']).set_index('event_time').sort_index()
daily = valid_time.resample('D').size().rename('events')
daily.plot(figsize=(14, 4), color='#3478a8', linewidth=1)
plt.title('Events per day (UTC)')
plt.xlabel('Date')
plt.ylabel('Events')
plt.tight_layout()
plt.show()

hourly_by_event = (valid_time.groupby([valid_time.index.hour, 'event'], observed=True)
                   .size().unstack(fill_value=0).reindex(range(24), fill_value=0))
hourly_by_event.plot(figsize=(12, 4), marker='o')
plt.title('Events by hour of day (UTC)')
plt.xlabel('Hour (UTC)')
plt.ylabel('Events')
plt.xticks(range(24))
plt.tight_layout()
plt.show()
display(daily.describe().to_frame('daily_event_count'))

## 8. Product funnel theo visitor
Đây là tỷ lệ visitor từng có event tương ứng trong toàn bộ file; không phải conversion theo session hay cohort. Transaction rows là proxy cho visitor đã mua ít nhất một item.

In [ ]:
visitor_event_flags = (df.assign(flag=1)
    .pivot_table(index='visitorid', columns='event', values='flag', aggfunc='max', fill_value=0, observed=True))
for label in ['view', 'addtocart', 'transaction']:
    if label not in visitor_event_flags.columns:
        visitor_event_flags[label] = 0
visitor_event_flags = visitor_event_flags[['view', 'addtocart', 'transaction']]
visitor_count = len(visitor_event_flags)
funnel = pd.DataFrame({
    'visitors': [int(visitor_event_flags[c].sum()) for c in visitor_event_flags.columns],
}, index=visitor_event_flags.columns)
funnel['share_of_all_visitors_pct'] = (funnel.visitors / visitor_count * 100).round(2)
display(funnel)
print('Visitors with cart / visitors with view:', round(funnel.loc['addtocart', 'visitors'] / max(funnel.loc['view', 'visitors'], 1), 4))
print('Visitors with transaction / visitors with cart:', round(funnel.loc['transaction', 'visitors'] / max(funnel.loc['addtocart', 'visitors'], 1), 4))

## 9. Item phổ biến theo event type
Transaction count ở đây là số transaction event rows theo item; tùy schema dữ liệu, một giao dịch có thể liên quan nhiều dòng item.

In [ ]:
top_items = (df.groupby(['event', 'itemid'], observed=True).size().rename('events')
             .reset_index().sort_values(['event', 'events'], ascending=[True, False]))
display(top_items.groupby('event', observed=True).head(10).reset_index(drop=True))

top_purchased = (df.loc[df.event == 'transaction'].groupby('itemid').size()
                 .sort_values(ascending=False).head(15).sort_values())
top_purchased.plot.barh(figsize=(8, 5), color='#a85c70')
plt.title('Top items by transaction event rows')
plt.xlabel('Transaction event rows')
plt.ylabel('Item ID')
plt.tight_layout()
plt.show()

## 10. Kết luận và câu hỏi tiếp theo
Sau khi chạy notebook, ghi lại các con số thực tế ở đây. Không điền kết luận trước khi xem output.

Gợi ý: event nào chiếm tỷ trọng lớn? Có visitor/item outlier không? Dataset bao phủ khoảng thời gian nào? Tỷ lệ cart → transaction theo visitor là bao nhiêu? Có khoảng trống hoặc burst đáng chú ý trên timeline không?

Nếu dùng dữ liệu này để đánh giá recommendation, cần chia train/test theo thời gian. Không dùng quan hệ co-view đã tạo từ tương lai của tập test để chấm điểm model.